# 11 · Hangi Modeli Ne Zaman Kullanmalı? (Kopya Kağıdı)

## Karar akışı
- **Etiketli veri var mı?**
  - **Evet → hedef sürekli mi?**
    - Evet → **REGRESYON:** Lineer → Ridge/Lasso → Random Forest → XGBoost/LightGBM
    - Hayır → **SINIFLANDIRMA:** Lojistik → Random Forest → XGBoost / SVM
  - **Hayır → amaç ne?**
    - Gruplama → **KÜMELEME:** K-Means, DBSCAN, Hiyerarşik
    - Boyut azaltma → **PCA**, t-SNE, UMAP
    - Aykırı bulma → **Isolation Forest**, LOF
  - **Zamana bağlı tahmin** → ARIMA/SARIMA, Prophet, lag özellikli XGBoost, LSTM
  - **Görüntü / metin / ses** → Derin öğrenme (CNN, Transformer)

## Model karşılaştırma tablosu

| Model | Tür | Yorumlanabilirlik | Ölçekleme? | Güçlü Yanı | Zayıf Yanı | Notebook |
|---|---|---|---|---|---|---|
| Lineer / Çoklu Lineer Reg. | Regresyon | ⭐⭐⭐ | Yorum için hayır | Hızlı, katsayılar anlamlı | Doğrusal olmayanı yakalamaz | 01 |
| Ridge / Lasso / ElasticNet | Regresyon | ⭐⭐⭐ | **Evet** | Overfitting & çoklu doğrusallığa dayanıklı | Doğrusal | 02 |
| Lojistik Regresyon | Sınıflandırma | ⭐⭐⭐ | Evet | Olasılık verir, baseline | Doğrusal sınır | 03 |
| KNN | İkisi de | ⭐⭐ | **Evet** | Basit | Büyük veride yavaş | 02, 03 |
| SVM | İkisi de | ⭐ | **Evet** | Yüksek boyutta güçlü | Büyük veride yavaş | 02, 03 |
| Naive Bayes | Sınıflandırma | ⭐⭐ | Hayır | Çok hızlı, metinde iyi | Bağımsızlık varsayımı | 03 |
| Karar Ağacı | İkisi de | ⭐⭐⭐ | Hayır | Görselleştirilebilir | Overfitting | 03 |
| Random Forest | İkisi de | ⭐⭐ | Hayır | Sağlam, az ayar ister | Büyük model | 04 |
| XGBoost / LightGBM | İkisi de | ⭐ | Hayır | Tablo veride en iyi | Çok hiperparametre | 04 |
| K-Means | Kümeleme | ⭐⭐ | **Evet** | Hızlı | k gerekli, küresel kümeler | 05 |
| DBSCAN | Kümeleme | ⭐⭐ | **Evet** | Gürültüye dayanıklı | eps seçimi zor | 05 |
| PCA | Boyut indirgeme | ⭐⭐ | **Evet** | Hızlı | Sadece doğrusal | 06 |
| Isolation Forest / LOF | Anomali | ⭐ | LOF için evet | Etiket gerektirmez | contamination tahmini | 07 |
| ARIMA / SARIMA | Zaman serisi | ⭐⭐ | Hayır | İstatistiksel temelli | Tek değişkenli, doğrusal | 08 |
| Sinir Ağları | İkisi de | ⭐ | **Evet** | Görüntü/metin/ses | Çok veri & hesaplama | 09 |

## Önerilen çalışma sırası
1. **Baseline** kur (ortalama / çoğunluk sınıfı / naif tahmin).
2. **Basit model** dene (Lineer / Lojistik Regresyon).
3. **Ağaç tabanlı ensemble** dene (Random Forest → XGBoost/LightGBM).
4. **Çapraz doğrulama** ile karşılaştır, **GridSearch/RandomizedSearch** ile ayarla.
5. Test setinde **bir kez** değerlendir, modeli kaydet.

## Altın kurallar
- ✅ Önce **train/test ayır**, sonra ön işle (`Pipeline` kullan).
- ✅ Eksik değerleri **analiz et**, sonra doldur/at — nedenini açıkla.
- ✅ Nominal değişken → **One-Hot**, sıralı değişken → **Ordinal** encoding.
- ✅ Mesafe/gradyan tabanlı modellerde **ölçekle**.
- ✅ Dengesiz sınıflarda accuracy yerine **F1 / Recall / PR-AUC**.
- ✅ Zaman serisinde **kronolojik** ayır.
- ✅ Basit model karmaşık modele yakın sonuç veriyorsa **basit olanı seç**.

In [ ]:
# Hızlı başlangıç şablonu – kendi veriniz için doldurun
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier   # veya RandomForestRegressor

def hizli_pipeline(df, hedef, model=RandomForestClassifier(random_state=42)):
    X, y = df.drop(columns=hedef), df[hedef]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    sayisal = X.select_dtypes("number").columns
    kategorik = X.select_dtypes(exclude="number").columns
    on_isleme = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), sayisal),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), kategorik),
    ])
    pipe = Pipeline([("on_isleme", on_isleme), ("model", model)]).fit(X_train, y_train)
    print("Test skoru:", round(pipe.score(X_test, y_test), 3))
    return pipe

# Kullanım:
# df = pd.read_csv("veriniz.csv")
# model = hizli_pipeline(df, hedef="hedef_sutun")